# PLOT-DAS walkthrough — residual-stream cells (RAVEL × Continent)

End-to-end run of the PLOT-DAS pipeline on **cell 22 — RAVEL × Gemma-2-2b × Continent**, our cleanest win (0.856 highest-view, +0.008 vs DAS LB 0.848). This is the residual-stream pipeline; for the attention-head pipeline used by IOI, see `ioi_walkthrough.ipynb`.

What this notebook shows:

1. Loading the per-task config (`PlotConfig` from `mib_submission/plot/configs.py`)
2. Building the `ExperimentBundle` (model + datasets + filter)
3. Running Stage A (layer OT) + Stage B (per-layer token-position OT) via `select_sites_via_plot`
4. Inspecting the picks and Sinkhorn plans
5. Pruning the bundle to picked sites only + training DAS rotations
6. Writing the MIB-format submission
7. Running the harness eval + reading per-site IIA

Reproducing via the CLI (equivalent to this notebook in one line):

```bash
.venv-mib/bin/python -m mib_submission.plot.run \
    --task ravel_task --model google/gemma-2-2b --variable Continent \
    --train-batch-size 16
.venv-mib/bin/python scripts/eval_cell.py \
    --cell ravel_task_Gemma2ForCausalLM_Continent
```

Runtime expectation: ~30 min on an 8 GB GPU (Gemma-2-2b at fp16). Set `--dataset-size 64` for a faster smoke.

In [ ]:
import sys
from pathlib import Path

# Run from notebooks/ or from repo root; both work.
REPO = Path.cwd() if (Path.cwd() / 'mib_submission').is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO))

import torch
import numpy as np
import matplotlib.pyplot as plt

from mib_submission.pipeline import setup_residual_experiment, MIB_TRACK
from mib_submission.plot.configs import default_config
from mib_submission.plot.pipeline import select_sites_via_plot
from mib_submission.site_keys import site_key_for_unit
from mib_submission.serialize import cell_folder_name

DTYPE = torch.float16
SUBMISSION_ROOT = REPO / 'submissions' / 'plot'
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Load the cell config

`default_config` returns a `RunConfig` containing:

- `plot_config: PlotConfig` — the V-row schema, alphabet source, signature dataset, OT solver settings, calibration sweep grids.
- DAS hyperparameters (`n_features`, `training_epochs`, `init_lr`, `train_batch_size`).
- LM-pipeline settings (`max_new_tokens`, `checker`).

For RAVEL Continent, the `_ravel_v3_attributes` preset configures:

- V=3 OT rows over `('Country', 'Continent', 'Language')`
- Alphabet derived from `causal_model.values['answer']` (~340 distinct attribute strings, compacted to ~263 first-token IDs under Gemma's tokenizer)
- `per_row_filter_attribute='queried_attribute'` — each row's signature collected only on bases where the queried attribute matches that row's variable
- `n_features=288`, 1 epoch DAS, `max_new_tokens=2` (multi-token answers like 'United States')

In [ ]:
config = default_config(
    task='ravel_task',
    model_name='google/gemma-2-2b',
    variable='Continent',
    overrides={'train_batch_size': 16},  # 8 GB VRAM guard
)

print(f'cell: {config.task} × {config.model_class_name} × {config.variable}')
print(f'plot_config.variables (OT rows): {config.plot_config.variables}')
print(f'plot_config.calibration_variable: {config.plot_config.calibration_variable}')
print(f'plot_config.per_row_filter_attribute: {config.plot_config.per_row_filter_attribute}')
print(f'plot_config.signature_dataset: {config.signature_dataset}')
print(f'plot_config.stage_a_top_k_grid: {config.plot_config.stage_a_top_k_grid}')
print(f'plot_config.stage_b_top_k_grid: {config.plot_config.stage_b_top_k_grid}')
print(f'plot_config.stage_a_epsilon_grid: {config.plot_config.stage_a_epsilon_grid}')
print(f'plot_config.stage_b_epsilon_grid: {config.plot_config.stage_b_epsilon_grid}')
print(f'DAS: n_features={config.n_features}, epochs={config.training_epochs}, batch={config.train_batch_size}')

## 2. Build the experiment bundle

`setup_residual_experiment` loads the LM pipeline, builds the causal model + counterfactual datasets, then runs the harness's `FilterExperiment` to keep only base examples the model already gets right under no intervention. The returned `ExperimentBundle` packages:

- `bundle.pipeline` — `LMPipeline` wrapping the HF model
- `bundle.causal_model` — the high-level causal abstraction (the V variables we'll interchange)
- `bundle.train_data`, `bundle.dev_data`, `bundle.test_data` — filtered counterfactual datasets per split
- `bundle.experiment` — pre-built `PatchResidualStream` with all `(layer × token_position)` candidate sites declared as `model_units_lists`

Gemma-2-2b: 26 layers × 3 token positions per layer = 78 candidate sites (some token positions per task).

In [ ]:
from transformers import AutoConfig
model_cfg = AutoConfig.from_pretrained(config.model_name)
layers = list(range(model_cfg.num_hidden_layers))
print(f'candidate layers: {layers} ({len(layers)} total)')

bundle = setup_residual_experiment(
    task=config.task,
    model_name=config.model_name,
    layers=layers,
    target_variables=[config.variable],
    dtype=DTYPE,
    dataset_size=config.dataset_size,
    config_overrides={
        'training_epoch': config.training_epochs,
        'init_lr': config.init_lr,
        'n_features': config.n_features,
        'batch_size': config.train_batch_size,
        'output_scores': False,
    },
    checker=config.checker,
    max_new_tokens=config.max_new_tokens,
    verbose=True,
)

print(f'\ntrain splits: {sorted(bundle.train_data.keys())}')
print(f'total candidate sites in bundle: {len(bundle.experiment.model_units_lists)}')

## 3. Run Stage A + Stage B (PLOT site selection)

`select_sites_via_plot` does the OT site-selection:

1. **Per-OT-row signature collection.** For each of the V variables (`Country`, `Continent`, `Language`), filter the signature dataset to bases where `queried_attribute == row_variable`, run the model with each (base, source) pair, and collect per-site output-probability deltas over the alphabet. Aggregate to per-(layer, position) rows. L2-row-normalize.
2. **Stage A.** Aggregate per-layer signatures across token positions (mean). Build abstract row table A (V rows of expected one-hot diffs under interchange). Compute `cost_matrix(A_normed, neural_normed)` and sweep `epsilon` × `top_k_per_row` over Sinkhorn-balanced solves. Per row, pick top-k layers. Union → Stage A picks.
3. **Stage B.** For each Stage-A-picked (row, layer), re-OT over the layer's available token positions. Pick top-k positions per (row, layer). Union → Stage B sites.
4. **Calibration sweep.** Across both stages, candidates are scored by mean per-site IIA on the calibration variable. The (epsilon, top_k) maximizing that mean is kept.

This is the same path called by `mib_submission.plot.run:main()` — we're just running it explicitly.

In [ ]:
fit_split = config.signature_dataset  # 'attribute_train' for RAVEL
print(f'running PLOT on split: {fit_split!r}\n')

selection = select_sites_via_plot(
    bundle,
    bundle.train_data[fit_split],
    config=config.plot_config,
    verbose=True,
)

a_eps, a_topk, a_score = selection.stage_a_chosen
b_eps, b_topk, b_score = selection.stage_b_chosen
print(f'\nStage A best: eps={a_eps} top_k={a_topk} IIA={a_score:.4f}')
print(f'Stage A picked layers: {selection.stage_a_layers}')
print(f'Stage B best: eps={b_eps} top_k={b_topk} IIA={b_score:.4f}')
print(f'\nStage B selected sites (layer, token_position):')
for s in selection.selected_sites:
    print(f'  {s}')

## 4. Visualize the Stage A Sinkhorn plan

The Sinkhorn plan π is a `(V, num_layers)` matrix: row `v` is the OT mass that variable `v` allocates across layers. We pick the top-k layers per row from this plan. A healthy plan concentrates mass on a few layers per row; a degenerate plan (uniform or near-uniform) means the OT signal was too weak — usually a sign of cardinality starvation or a bad alphabet.

For Continent (6 distinct values, ~43 examples per value at ds=256), we expect a moderately concentrated plan. For Country / Language (160-174 values, ~1.5 examples per value), the plan would look noisier.

In [ ]:
pi = selection.stage_a_pi  # (V, num_layers) tensor
candidate_layers = sorted({
    site_key_for_unit(mul[0][0])[0]
    for mul in bundle.experiment.model_units_lists
})

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(pi.numpy(), aspect='auto', cmap='viridis')
ax.set_yticks(range(len(config.plot_config.variables)))
ax.set_yticklabels(config.plot_config.variables)
ax.set_xticks(range(len(candidate_layers)))
ax.set_xticklabels([f'L{l}' for l in candidate_layers], rotation=90, fontsize=7)
ax.set_title(f'Stage A Sinkhorn plan π (V × layers) — Continent cell')
ax.set_xlabel('layer')
ax.set_ylabel('OT row (variable)')
plt.colorbar(im, ax=ax, label='mass')
plt.tight_layout()
plt.show()

# Per-row top picks
for v_idx, v_name in enumerate(config.plot_config.variables):
    row = pi[v_idx]
    top_idxs = torch.argsort(row, descending=True)[:5]
    picks = [(candidate_layers[i], float(row[i])) for i in top_idxs]
    print(f'{v_name:12s} top 5: ' + '  '.join(f'L{L}={m:.3f}' for L, m in picks))

## 5. Prune the bundle and train DAS at picked sites

PLOT's contribution ends here. From this point on we hand off to MIB's `train_interventions(method="DAS")` which trains orthogonal-rotation featurizers via gradient descent at each picked site. We *prune* the bundle to the picked sites only — DAS would otherwise train at every (layer, token_position) candidate (72 sites for Gemma residual stream), which is exactly what the baseline-DAS approach does and what PLOT's site selection avoids.

In [ ]:
selected_set = set(selection.selected_sites)
pruned = [
    mul for mul in bundle.experiment.model_units_lists
    if site_key_for_unit(mul[0][0]) in selected_set
]
print(f'Bundle had {len(bundle.experiment.model_units_lists)} sites; pruning to {len(pruned)}.')
bundle.experiment.model_units_lists = pruned

print(f'\nTraining DAS at {len(pruned)} sites for {config.training_epochs} epoch(s)...')
bundle.experiment.train_interventions(
    bundle.train_data,
    [config.variable],
    method='DAS',
    verbose=True,
)

## 6. Write the submission folder

MIB expects each picked site to ship as 3 files: `<site_id>_featurizer`, `<site_id>_inverse_featurizer`, `<site_id>_indices`. `save_featurizers` writes them to the submission folder. `verify_submission.py` checks format afterward.

In [ ]:
import shutil
import subprocess

cell = cell_folder_name(config.task, config.model_class_name, config.variable)
cell_dir = SUBMISSION_ROOT / cell
if cell_dir.exists():
    shutil.rmtree(cell_dir)
cell_dir.mkdir()
bundle.experiment.save_featurizers(None, str(cell_dir))

n_files = sum(1 for _ in cell_dir.iterdir())
print(f'wrote {n_files} files to {cell_dir}')

# Verify
result = subprocess.run(
    [sys.executable, str(MIB_TRACK / 'verify_submission.py'), str(SUBMISSION_ROOT)],
    capture_output=True, text=True,
)
print('\n' + result.stdout.strip().split('\n')[-1])

## 7. Run the harness eval

The MIB harness's `evaluate_submission_task` loads the shipped featurizers, runs interventions on the public test splits, and writes a results JSON into the cell folder.

We use `scripts/eval_cell.py` rather than calling the harness directly — it applies two patches the harness needs but doesn't ship:

1. **`max_new_tokens` override per task.** The harness hardcodes `LMPipeline(..., max_new_tokens=1)` for all tasks; RAVEL's multi-token answers (`'United States'`) need `max_new_tokens=2`.
2. **`LMPipeline.load` `position_ids` fallback** for transformers 5.x compatibility (needed for IOI; harmless for RAVEL).

Both patches are idempotent (re-running is a no-op).

In [ ]:
result = subprocess.run(
    [sys.executable, str(REPO / 'scripts' / 'eval_cell.py'),
     '--cell', cell],
    capture_output=True, text=True,
)
# Print the last few lines of eval output
for line in result.stdout.strip().split('\n')[-10:]:
    print(line)

## 8. Inspect the per-site IIA breakdown

The eval writes per-(split, site) IIA scores into the cell folder. MIB's leaderboard uses the **highest-view** aggregation: per-site avg across splits → per-layer max across sites → max across layers.

In [ ]:
import json, collections

results_files = list(cell_dir.glob('*results.json'))
d = json.loads(results_files[0].read_text())
splits = list(d['dataset'].keys())
sites = list(d['dataset'][splits[0]]['model_unit'].keys())

print(f'splits: {splits}\n')
print(f'{"site":52s} | ' + ' | '.join(f'{s[:18]:18s}' for s in splits) + ' | per-site avg')
print('-' * 130)

by_layer = collections.defaultdict(list)
for site in sites:
    short = site.split('Token-')[1].rstrip("')]")
    layer = d['dataset'][splits[0]]['model_unit'][site]['metadata']['layer']
    scores = []
    for s in splits:
        sv = d['dataset'][s]['model_unit'][site]
        for k, v in sv.items():
            if isinstance(v, dict) and 'average_score' in v:
                scores.append(v['average_score']); break
    avg = sum(scores)/len(scores)
    by_layer[layer].append(avg)
    print(f'  L{layer:2d} {short:48s} | ' + ' | '.join(f'{x:18.4f}' for x in scores) + f' | {avg:.4f}')

highest_view = max(max(vs) for vs in by_layer.values())
average_view = sum(max(vs) for vs in by_layer.values()) / len(by_layer)
print(f'\nhighest-view: {highest_view:.4f}')
print(f'average-view: {average_view:.4f}')
print(f'\nreference: DAS leaderboard 0.848, our shipped 0.856')

## Per-task config differences

The above pipeline works unchanged for MCQA, ARC, arithmetic by swapping the `PlotConfig` preset. The 1–2 line changes per task:

| task | preset function | key non-default fields |
|---|---|---|
| MCQA | `_mcqa_v4_choices` | `variables=('choice0', 'choice1', 'choice2', 'choice3')`, `letters='ABCDEFGHIJKLMNOPQRSTUVWXYZ'`, `signature_dataset='answerPosition_randomLetter_train'` |
| ARC | `_arc_v4_symbols` | `variables=('symbol0', 'symbol1', 'symbol2', 'symbol3')`, same alphabet/signature, `stage_b_top_k_grid=(1,)` (per PLOT_SHORTCOMINGS §8) |
| arithmetic | `_arithmetic_v2_carry_children` | `variables=('tens_out', 'hundreds_out')` (non-target OT rows to avoid V=1 collapse), `letters='0123456789'`, `output_key='raw_output'`, `label_from_output=_arithmetic_first_digit`, `max_new_tokens=3` |
| RAVEL | `_ravel_v3_attributes` | `variables=('Country', 'Continent', 'Language')`, `answer_alphabet_from_causal_model=True`, `per_row_filter_attribute='queried_attribute'`, `max_new_tokens=2`, custom `_ravel_checker` |

All four use `setup_residual_experiment` and `select_sites_via_plot`. IOI is structurally different — see `ioi_walkthrough.ipynb`.

## Further reading

- `WALKTHROUGHS.md` — prose version of this notebook covering all 5 tasks
- `PLOT_SHORTCOMINGS.md` — diagnosed limits, including the DAS-vs-identity finding (§15) that makes cell 8 land at 0.999
- `JOURNAL.md` — methodological narrative, including the cell-22 Continent smoke→full journey